In [1]:
import torch
import os
import numpy as np
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import shufflenet_v2_x1_0
from sklearn.model_selection import train_test_split
import torch.nn as nn

# FX Quantization imports
from torch.ao.quantization import get_default_qconfig, QConfigMapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx


torch.backends.quantized.engine = 'fbgemm'
device = torch.device("cpu")  # PTQ runs/evaluates on CPU

# Dataset & DataLoaders 
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
print(f"Total images: {len(dataset)}")
print(f"Number of classes: {len(dataset.classes)}")
print("Class names (subfolder labels):", dataset.classes[:10], "...")

# Stratified Split
from sklearn.model_selection import train_test_split
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []

for class_id in np.unique(targets):
    idxs = np.where(targets == class_id)[0]
    # first split train vs temp (val+test)
    train_ids, temp_ids = train_test_split(
        idxs, test_size=0.15, random_state=42, shuffle=True
    )
    # then split temp into val vs test
    val_ids, test_ids = train_test_split(
        temp_ids, test_size=1/3, random_state=42, shuffle=True
    )
    train_idx.extend(train_ids)
    val_idx.extend(val_ids)
    test_idx.extend(test_ids)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# Subsets 
train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

# DataLoaders 
batch_size = 128
num_workers = min(8, os.cpu_count())

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)

print("DataLoaders ready")

# Load trained FP32 model
state_dict = torch.load("/home/h5/sapi279d/shufflenetv2_weights.pth", map_location="cpu")

# Remove the "_orig_mod." prefix from keys
new_state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

num_classes = 200
model_fp32 = shufflenet_v2_x1_0(weights=None)
model_fp32.fc = nn.Linear(model_fp32.fc.in_features, num_classes)

model_fp32.load_state_dict(new_state_dict, strict=True)
model_fp32.eval()
print("Weights loaded successfully")

# FX Graph Mode PTQ 
torch.backends.quantized.engine = "fbgemm"

qconfig = get_default_qconfig("fbgemm")
qconfig_mapping = QConfigMapping().set_global(qconfig)

example_inputs = (torch.randn(1, 3, 224, 224),)

# Prepare model for quantization (inserts Quant/DeQuant)
prepared = prepare_fx(model_fp32, qconfig_mapping, example_inputs)

# Calibration: run a few batches through the model
def calibrate_fx(model, loader, num_batches=20):
    model.eval()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches:
                break
            model(images)

calibrate_fx(prepared, train_loader, num_batches=20)
print("Calibration done (FX)")

# Convert to INT8
model_int8 = convert_fx(prepared)
model_int8 = model_int8.to("cpu")
print("Static quantization complete (FX)")

# Evaluation
def evaluate_model(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            outputs = model(images)  # float inputs allowed (FX handles Q/DQ)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

acc_fp32 = evaluate_model(model_fp32, test_loader)
acc_int8 = evaluate_model(model_int8, test_loader)

print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"INT8 Accuracy: {acc_int8:.2f}%")


Total images: 100000
Number of classes: 200
Class names (subfolder labels): ['n01443537', 'n01629819', 'n01641577', 'n01644900', 'n01698640', 'n01742172', 'n01768244', 'n01770393', 'n01774384', 'n01774750'] ...
Train: 85000, Val: 10000, Test: 5000
DataLoaders ready ✅
✅ Weights loaded successfully


/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torch/ao/quantization/observer.py:220: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(
/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torch/cuda/__init__.py:141: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 802: system not yet initialized (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


Calibration done ✅ (FX)
Static quantization complete ✅ (FX)
FP32 Accuracy: 49.98%
INT8 Accuracy: 49.58%


In [2]:
import time

def get_model_size(model, filename="temp.pth"):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / 1e6
    os.remove(filename)
    return size_mb

def benchmark(model, loader, num_batches=50):
    model.eval()
    start = time.time()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches:
                break
            _ = model(images)
    end = time.time()
    total_images = num_batches * loader.batch_size
    total_time = end - start
    return (total_time / num_batches) * 1000,  (total_time / total_images) * 1000  # ms/batch, ms/image

# Model size
fp32_size = get_model_size(model_fp32)
int8_size = get_model_size(model_int8)

print(f"FP32 model size: {fp32_size:.2f} MB")
print(f"INT8 model size: {int8_size:.2f} MB")

# Runtime (CPU only, first 50 batches) 
batch_ms_fp32, img_ms_fp32 = benchmark(model_fp32, test_loader)
batch_ms_int8, img_ms_int8 = benchmark(model_int8, test_loader)

print(f"FP32 Inference: {batch_ms_fp32:.2f} ms/batch | {img_ms_fp32:.2f} ms/image")
print(f"INT8 Inference: {batch_ms_int8:.2f} ms/batch | {img_ms_int8:.2f} ms/image")


FP32 model size: 6.00 MB
INT8 model size: 1.72 MB
FP32 Inference: 648.63 ms/batch | 5.07 ms/image
INT8 Inference: 2690.70 ms/batch | 21.02 ms/image
